In [1]:
!pip install -q groq gradio

In [6]:
import gradio as gr
from groq import Groq
from google.colab import userdata

client = Groq(api_key="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")
MODEL = "openai/gpt-oss-120b"

SYSTEM_PROMPT = "You are a helpful, friendly assistant. Use earlier messages in the conversation for context. Keep answers clear and concise."

def chat(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for h in history[-10:]:  # last 10 turns keeps it within token limits
        if isinstance(h, dict):          # new Gradio: {"role", "content"}
            messages.append({"role": h["role"], "content": h["content"]})
        else:                            # old Gradio: [user_msg, bot_msg]
            if h[0]:
                messages.append({"role": "user", "content": h[0]})
            if h[1]:
                messages.append({"role": "assistant", "content": h[1]})

    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        stream=True,
    )

    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        yield reply

gr.ChatInterface(
    fn=chat,
    title="Context-Aware AI Assistant",
    description="Chat with memory of the current conversation.",
).launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bafe864e9695ff3d68.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://bafe864e9695ff3d68.gradio.live
